# 4.1 Constructing an evanescent field

The integer phase rule continues beyond the set of real outgoing directions. An order outside that set requires a complex normal wavevector component, giving a field that varies along the grating and decays away from it.

 For these direction constructions the incident and output media have the same index; refraction between unequal exterior media is not simulated.

[Notebook three](3_0_Why_Diffraction_Orders_Appear.ipynb) connected diffraction orders to an integer number of extra phase cycles per grating period relative to the incident wave. Its wave plots showed only candidates with real outgoing directions. Here we follow one of the excluded candidates and construct its field.

The construction follows a missing order through its normal wavevector component to a real, decaying field. [Lesson 4.2](4_2_Decay_Length_Grazing_and_Energy_Flow.ipynb) develops decay length, grazing, and energy flow.

Run the cells from top to bottom in a Jupyter kernel with NumPy, SciPy, Matplotlib, Plotly and ipywidgets. This lesson runs independently and its explorer keeps evanescent orders selectable. Each **small calculation** has its own editable parameters; changing a calculation does not move the explorer controls.

Read the prediction, run the cell, and then open the answer.

We retain the earlier geometry: the grating lies at $z=0$, repeats along $x$ with period $d$, and has grooves along $y$. Incidence is from $z>0$; the outgoing side is $z<0$. Angles are measured from $-z$ toward $+x$, and $k_{y,i}=k_{y,m}=0$. Both sides have the same homogeneous, isotropic, lossless medium. $\lambda$ is the wavelength **in that medium**, with lengths in micrometres.

Use $\mathbf k_i$ for the incident wavevector and $\mathbf k_m=(k_{x,m},0,k_{z,m})$ for diffraction order $m$. Plain $k=2\pi/\lambda$ is the positive scalar wavenumber of the medium; it is distinct from the vector $\mathbf k_m$.

The field examples prescribe **one component with surface amplitude $A=1$**. They show what that component would do if excited. They do not determine which amplitudes a particular grating generates or combine the components into a full scattering solution.

In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
from IPython.display import display, Image

# Normal imports avoid reloading helper modules on each run.


## 1. Keep an order that notebook three could not draw

Periodicity requires

$$k_{x,m}=k_{x,i}+mG,\qquad G=\frac{2\pi}{d},\qquad m\in\mathbb Z,$$

where $m$ is the diffraction-order number. This is a condition on phase variation **along the surface**. The condition remains meaningful even when the corresponding field has no real outgoing angle. The separation between direction and strength is also discussed in [Ansys, “Grating physics”](#ref-4-1-ansys) [\[1\]](#ref-4-1-ansys).

Use notebook three's starting values:

$$\lambda=d=2\,\mu\mathrm m,\qquad \theta_i=30^\circ,\qquad
k=\pi\,\mathrm{rad}/\mu\mathrm m,\qquad \frac{k_{x,i}}k=\frac12.$$

Then

$$\frac{k_{x,m}}k=\frac{k_{x,i}}k+m\frac{G}{k}
=\frac12+m\frac{\lambda}{d}=\frac12+m.$$

| Order $m$ | $k_{x,m}/k$ | Real outgoing direction? |
| --- | --- | --- |
| $-2$ | $-1.5$ | No |
| $-1$ | $-0.5$ | Yes: $-30^\circ$ |
| $0$ | $+0.5$ | Yes: $+30^\circ$ |
| $+1$ | $+1.5$ | No |

The earlier dropdown listed $m=-1$ and $m=0$. We will start by selecting **$m=+1$** instead.

**Predict:** does $m=+1$ fail the integer phase condition? Or does it satisfy that condition but fail a separate condition for a real direction?

:::{dropdown} Answer

It satisfies the integer phase condition: $(k_{x,m}-k_{x,i})d=2\pi$. The difficulty is that $k_{x,m}=1.5k$. A real vector of length $k$ cannot have a horizontal component longer than $k$.

:::

- For propagating and grazing orders, $k_{x,m}/k=\sin\theta_m$ is the $x$ direction cosine; recall that $\theta_m$ is measured from $-z$.
- For an evanescent order, $|k_{x,m}/k|>1$ remains a valid normalized horizontal wavevector component, but no real angle has that sine.
- The real-vector triangle from the earlier notebooks therefore cannot represent this evanescent component.

### Small calculation 1: list the candidates before filtering them

Compute the normalized horizontal component and the remaining squared normal component. A positive result permits propagation away from the grating; zero is grazing; a negative result is evanescent. This calculation prints the candidates instead of deleting the evanescent ones.

**Predict and rerun:** keep $\lambda=d$ but change incidence to $0^\circ$. What happens to $m=\pm1$? The small tolerance below treats roundoff at grazing as zero.


In [ ]:
wavelength, d = 2.0, 2.0  # micrometres
theta_i_deg = 30.0
orders = np.arange(-2, 3)
alpha = np.sin(np.deg2rad(theta_i_deg)) + orders*wavelength/d
radicand = 1 - alpha**2  # k_mz**2 / k**2

print(" m    k_mx/k  k_mz²/k²   classification")
for m, a, r in zip(orders, alpha, radicand):
    state = "grazing" if abs(r) <= 1e-12 else ("propagating" if r > 0 else "evanescent")
    print(f"{m:+2d}   {a:+7.3f}   {r:+7.3f}    {state}")


:::{dropdown} Answer for the starting values and the suggested change

At $30^\circ$, the normalized horizontal components for $m=-2,-1,0,+1,+2$ are $-1.5,-0.5,+0.5,+1.5,+2.5$. Only $m=-1,0$ are propagating.

At normal incidence, $k_{x,m}/k=m$. The zero order propagates, $m=\pm1$ are grazing, and $m=\pm2$ are evanescent. A grazing wave travels along the grating; it does not travel into the outgoing half-space.

:::


## 2. Complete the normal component

For a field component in the uniform medium, the wave equation requires

$$k_{x,m}^2+k_{z,m}^2=k^2,
\qquad\text{so}\qquad
k_{z,m}^2=k^2-k_{x,m}^2.$$

In the outgoing half-space $z<0$, choose the following branches:

| Condition | Normal component $k_{z,m}$ | Behaviour with distance from the grating |
| --- | --- | --- |
| $\lvert k_{x,m}\rvert <k$ | $-\sqrt{k^2-k_{x,m}^2}$ | Phase advances as the wave travels away |
| $\lvert k_{x,m}\rvert =k$ | $0$ | Grazing: no normal phase change or decay |
| $\lvert k_{x,m}\rvert >k$ | $-i\kappa_m$, with $\kappa_m=\sqrt{k_{x,m}^2-k^2}>0$ | Amplitude decays away |

Here $i$ is the imaginary unit, defined by $i^2=-1$. For the last row, $(-i\kappa_m)^2=-\kappa_m^2$, providing the required negative square.

For real propagating or grazing vectors, the equation is also the familiar length condition $|\mathbf k_m|=k$. **Once a component is imaginary, use the algebraic squares shown above.** In vector form this is $\mathbf k_m\cdot\mathbf k_m=k^2$, with no complex conjugation. It is not an ordinary length statement for a complex vector.
- Replacing $k_{z,m}^2$ with $|k_{z,m}|^2$ would change the equation; $k$ still denotes the positive wavenumber of the medium.

The normal-component relation and the outgoing branch are developed in [TU Delft, “Angular Spectrum Method,” Eq. (375)](#ref-4-1-delft-diffraction) [\[2\]](#ref-4-1-delft-diffraction). That source uses propagation toward $+z$; our outgoing side is $-z$, so the signs of the normal components are reversed.

:::{dropdown} Optional: where does the equation come from?

The scalar Helmholtz equation in our two-dimensional uniform region is

$$\frac{\partial^2 U}{\partial x^2}+\frac{\partial^2 U}{\partial z^2}+k^2U=0.$$

Try a separated field $U=Ae^{i(k_{x,m}x+k_{z,m}z)}$. Each derivative in $x$ brings down $ik_{x,m}$, so the second derivative is $-k_{x,m}^2U$. The corresponding result in $z$ is $-k_{z,m}^2U$. Substituting gives

$$(-k_{x,m}^2-k_{z,m}^2+k^2)U=0.$$

For a nonzero component, the expression in parentheses must vanish. This derivation works with real or complex components; it never introduced a complex conjugate.

:::

## 3. An imaginary component produces a real, decaying field

Keep the time convention from the earlier notebooks:

$$u_m(x,z,t)=\operatorname{Re}\{U_m(x,z)e^{-i\omega t}\},\qquad
U_m(x,z)=A e^{i(k_{x,m}x+k_{z,m}z)}.$$

The capital $U_m$ is the time-independent **complex scalar field**. The lower-case $u_m$ is the real, signed field that we plot. For these examples, $A>0$ is the amplitude at the surface and the initial phase is zero.

For an evanescent order, insert $k_{z,m}=-i\kappa_m$:

$$e^{ik_{z,m}z}=e^{i(-i\kappa_m)z}=e^{\kappa_m z}.$$

Since the outgoing side has $z<0$, this factor gets **smaller** farther below the surface. It is helpful to name the positive distance into that side:

$$s=-z\geq0.$$

The field becomes

$$\boxed{u_m(x,-s,t)=A e^{-\kappa_m s}
\cos(k_{x,m}x-\omega t).}$$

There are two separate factors to read:

- $e^{-\kappa_m s}$ is the **amplitude envelope**. It decreases with distance and is independent of time.
- $\cos(k_{x,m}x-\omega t)$ oscillates along the surface and in time. At a fixed $x$, its phase is the same at every depth.

This is the meaning of *evanescent in the normal direction*. The field is not identically zero beyond the surface, and it still oscillates. The alternative branch $k_{z,m}=+i\kappa_m$ would grow as $s$ increases without limit; it does not describe an isolated grating driving an unbounded outgoing region.

**Predict:** at $x=0$ and $t/T=1/4$, is a zero signed field proof that the amplitude envelope has vanished?

### Small calculation 2: distinguish time from distance

Use the excluded $m=+1$ order from section 1. First evaluate its complex field $U_m$ at three depths. Then multiply by the time factor and take the real part, exactly as in the boxed expression. In Python, `1j` represents $i$, and `np.abs(U_m)` gives the local amplitude.

**Predict and rerun:** change only `tau` from $0$ to $0.25$, then $0.5$. Which printed column should remain unchanged?


In [ ]:
wavelength, d = 2.0, 2.0  # micrometres
theta_i_deg, m = 30.0, 1
tau = 0.0  # t/T; try 0.25, then 0.5
k = 2*np.pi/wavelength
k_x_i = k*np.sin(np.deg2rad(theta_i_deg))
k_x_m = k_x_i + m*2*np.pi/d

if abs(k_x_m) <= k:
    print("This calculation needs an evanescent order: choose |k_x_m| > k.")
else:
    kappa_m = np.sqrt(k_x_m**2-k**2)
    k_z_m = -1j*kappa_m
    s = np.array([0.0, 0.25, 0.5])  # positive distance into z < 0
    x = 0.0
    U_m = np.exp(1j*(k_x_m*x + k_z_m*(-s)))  # complex field; surface amplitude A = 1
    field = np.real(U_m*np.exp(-2j*np.pi*tau))
    print(f"kappa_m = {kappa_m:.3f} per micrometre; k_z_m = {k_z_m:.3f}")
    print("   s (um)    |U|/A     u/A")
    print(np.round(np.column_stack([s, np.abs(U_m), field]), 3))


:::{dropdown} Answer

Here $\kappa_m=\pi\sqrt{1.25}\approx3.512\,\mu\mathrm m^{-1}$. At $s=0,0.25,0.5\,\mu\mathrm m$, the amplitudes are approximately $1,0.416,0.173$.

At $t/T=0$, the signed field at $x=0$ equals those amplitudes. At $t/T=1/4$, it is zero at all three depths, to numerical precision. At $t/T=1/2$, it equals the negatives of those amplitudes. **The amplitude column does not change with time.**

In the spatial plot, other $x$ positions can still have nonzero signed fields when the entire $x=0$ depth cut is zero.

:::


## 4. Explore the field of the selected order

The explorer starts with $\lambda=d=2\,\mu\mathrm m$, $\theta_i=30^\circ$, and the previously excluded order $m=+1$.

- **Top ruler:** a sample of the integer candidates, $m=-3,\ldots,+3$, placed at $k_{x,m}/k$. The shaded band spans $-1$ to $+1$; its edges are grazing. A ring marks the selected order. This ruler is a component axis, not a drawing of the grating.
- **Left plot:** the selected signed field below $z=0$. Red and blue mean opposite signs; the same linear scale, $-1$ to $+1$, is used in every state. The dotted vertical line is $x=0$.
- **Right plot:** the signed field along that dotted line (blue dashed curve), together with its positive and negative amplitude envelope (orange). Its horizontal coordinate is the positive distance $s=-z$.
- **Readout:** the selected order's component and classification. For evanescent orders it also gives the amplitude decay length $\ell=1/\kappa_m$, explained in Notebook [4.2](4_2_Decay_Length_Grazing_and_Energy_Flow.ipynb).

**Try these in order:**

The decay is normal to the surface. Along x, this ideal component remains an extended sinusoidal field; the image frame does not give it a finite beam width. Decay in the lossless outgoing medium is not material absorption. Surface amplitude is prescribed, so a real grating’s excitation strength and a near-field probe’s response would need separate models.
- See the existing [lossless evanescent-field explanation](#ref-4-1-delft-basics) [\[3\]](#ref-4-1-delft-basics). 

1. Start with $m=+1$. Move **Time t/T**, or press Play. The stripes move along $x$ while their contrast decreases with depth. The orange envelope remains fixed.
2. Set $t/T=0.25$. At $x=0$, the blue depth curve is zero. Compare it with the orange envelope and with other $x$ positions in the field.
3. Select $m=0$ with the same illumination. Its amplitude does not decay, and the signed field generally changes phase with depth.
4. Restore **[Chapter 3](3_0_Why_Diffraction_Orders_Appear.ipynb) example**, then select $m=+2$. Does the field reach as far below the surface?
5. Return to $m=+1$. Change wavelength or period until the classification changes. **The order remains selected**, even when it is evanescent.

**Depth shown** zooms both spatial axes, preserving the physical aspect ratio: the horizontal range extends from minus that depth to plus that depth. It changes the view, not the field. Use a shallow window to inspect rapid decay. The seven displayed order numbers are a sample, not a count of all available orders.
- The three state-example buttons set normal incidence and the parameters used in Notebook [4.2](4_2_Decay_Length_Grazing_and_Energy_Flow.ipynb); **[Chapter 3](3_0_Why_Diffraction_Orders_Appear.ipynb) example** restores the tilted starting case.

In [ ]:
# Rerunning this cell closes the old explorer before creating its replacement.
if 'evanescent_explorer' in globals():
    evanescent_explorer.close()

import wave_optics.evanescent as evanescent
import wave_optics.evanescent_view as evanescent_view

evanescent_explorer = evanescent_view.EvanescentOrderExplorer()
display(evanescent_explorer.ui)


## Checkpoint

1. Does an evanescent order satisfy the integer-order rule?
2. Why is $k_{z,m}=-i\kappa_m$ the decaying branch for $z<0$?
3. Can an instantaneous zero field have a nonzero envelope?

:::{dropdown} Answers

It satisfies the tangential phase-repeat condition. Its normal factor is $e^{\kappa_m z}=e^{-\kappa_m s}$, which decreases into $z<0$. The oscillating cosine can be zero while its amplitude envelope remains nonzero.

:::

Continue to [4.2: decay length, grazing, and energy flow](4_2_Decay_Length_Grazing_and_Energy_Flow.ipynb).

## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete [scripts download](Downloads.md) to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. Complete the normal component

**Script:** [4_1_01_complex_longitudinal_root.py](Wave_solver_py_scripts/scripts/chapter_4/4_1_01_complex_longitudinal_root.py)

Compute propagating and evanescent longitudinal components.

### 2. An imaginary component produces a real decaying field

**Script:** [4_1_02_construct_decaying_field.py](Wave_solver_py_scripts/scripts/chapter_4/4_1_02_construct_decaying_field.py)

Construct and sample an evanescent field in space.

### 3. Constructing an evanescent solution

**Script:** [4_1_03_evanescent_helmholtz_residual.py](Wave_solver_py_scripts/scripts/chapter_4/4_1_03_evanescent_helmholtz_residual.py)

Verify that a decaying oscillatory field still satisfies Helmholtz.

### 4. Keep an order that has no real outgoing direction

**Script:** [4_1_04_mixed_spectrum_propagation.py](Wave_solver_py_scripts/scripts/chapter_4/4_1_04_mixed_spectrum_propagation.py)

Propagate a boundary containing both real and imaginary longitudinal components.




(refs-4-1)=

## References

(ref-4-1-ansys)=

[1] Ansys Optics, “Grating projections in FDTD overview,” “Grating physics.” [Online]. Available: [source](https://optics.ansys.com/hc/en-us/articles/360034394354-Grating-projections-in-FDTD-overview).

**Reading notes:** discrete tangential components and the distinction between directions and scattering strengths.

(ref-4-1-delft-diffraction)=

[2] A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, “Scalar Diffraction Optics,” TU Delft. [Online]. Available: [TU Delft, “Angular Spectrum Method,” Eq. (375)](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html#angular-spectrum-method).

**Reading notes:** Eq. (375), the outgoing square-root branch, and the propagating/evanescent distinction. Its $+z$ outgoing convention is the reverse of this notebook's $-z$ convention; here $\lambda$ always means wavelength in the medium.

(ref-4-1-delft-basics)=

[3] A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, “Basic Electromagnetic and Wave Optics,” TU Delft. [Online]. Available: [source](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap1_Basics/Basics_2022_01Clean.html#total-internal-reflection-and-evanescent-waves).

**Reading notes:** decaying fields and energy flow parallel to an interface in a lossless medium.

## Further Reading

(ref-4-1-absorbing-evanescent)=

- S. Anantha Ramakrishna and A. D. Armour, “Propagating and evanescent waves in absorbing media,” *American Journal of Physics*, vol. 71, p. 562, 2003. [Online]. Available: [preprint](https://arxiv.org/abs/cond-mat/0206110). doi: 10.1119/1.1557301.

  **Reading notes:** **Access:** Free arXiv manuscript; not public domain. **Read for:** Optional extension: distinguish evanescence from absorption; material model differs.

(ref-4-1-maystre)=

- D. Maystre, “Analytic Properties of Diffraction Gratings,” in *Gratings: Theory and Numeric Applications*, 2nd revisited ed., E. Popov, Ed. Institut Fresnel, 2014, ch. 2. [Online]. Available: [PDF](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=33).

  **Reading notes:** **Access:** Free institutional book chapter; not public domain. **Read for:** Read Sections 2.3–2.5: order expansion and energy balance.

## Reference notes

To release the explorer's controls and stop playback when finished, run `evanescent_explorer.close()`. Rerunning its creation cell already does this before replacing it.

Freely accessible works are not necessarily in the public domain.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.



## Course connections

[Chapter 3 — Phase change and diffraction orders](3_0_Why_Diffraction_Orders_Appear.ipynb): the origin of the integer condition used here.

- [5.1: three-dimensional orders](5_1_Three_Dimensional_Orders_and_Conical_Diffraction.ipynb) and [5.2: coordinate representations](5_2_Direction_Cosines_and_Spatial_Frequency_Coordinates.ipynb) extend the geometry using the same outgoing half-space. Sources using outgoing $+z$ require translating the longitudinal sign convention.
